Mostly summary from https://github.com/avito-tech/applied_statistics/blob/main/Ratio%20metrics.ipynb 

In [46]:
import numpy as np
import pandas as pd
import random
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sc
import math
from scipy import stats 
from scipy.optimize import minimize
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint

In [47]:
np.random.seed(42)
random.seed(42)

Ratio metrics are metrics of the following form:

$$ \overline{X} / \overline{Y} $$

It's difficult to test hypotheses on RM because iid conditions usually are not satisfied

In this case we will check CR of groups by unpacking data into 0, 1 array

In [48]:
def generate_sample(size=10):
    stage_1 = stats.norm(loc=10, scale=1).rvs(size=size)
    stage_2 = stats.uniform(0, 1).rvs(size=size)

    conv = np.floor(stage_1 * stage_2)
    sum = np.sum(conv)

    return [1] * int(sum) + [0] * int(np.sum(stage_1) - sum)

sample = generate_sample(10)
print(len(sample))

104


In [49]:
#Stage_1 is drawn from normal distribution, so we can use t-test

alpha = 0.05
group_size = 500
cnt = 0
n = 1000

for i in range(n):
    treatment = generate_sample(size=group_size)
    control = generate_sample(size=group_size)
    pval = stats.ttest_ind(treatment, control, equal_var=True, alternative='two-sided').pvalue
    cnt += (pval <= alpha)
print("FPR:", cnt / n)


FPR: 0.285


Sample is not independent because 10 users generate ~100 observations based on their conversion rate, while t-test thinks that sample came from 100 independent people (not iid). 


Thus FPR is enormous

Now we want to compare CR in different groups. We can use bootstrap to do that.
$$ H_0:\frac{P^c_{stage_2}}{P^c_{stage_1}}-\frac{P^t_{stage_2}}{P^t_{stage_1}}=0 $$

In [50]:
def generate_group_sample(group_size):
        
    O_array = stats.norm(loc=50, scale=5).rvs(size=group_size).astype(int)
    p_array = stats.uniform().rvs(size=group_size)
    
    M_array = (O_array * p_array).astype(int)
    
    return {
        "O_array": O_array,
        "message_array": M_array
    }

Bootstraping indexes to preserve user info

In [51]:
def ratio_bootstrap(message_array, message_control, n_test, n_control, alpha=0.05):
    theta_func = lambda MT, MC, OT, OC: np.sum(MT, axis=1) / np.sum(OT, axis=1) - np.sum(MC, axis=1) / np.sum(OC, axis=1)
    
    B = 1000
    batch_size = B // 20

    
    theta_asterisk_array = []

    test_size = len(message_array)
    control_size = len(message_control)

    test_inds_array = np.arange(0, test_size)
    control_inds_array = np.arange(0, control_size)
    for _ in range(0, B, batch_size):
        
        boot_t_inds = np.random.choice(test_inds_array, replace=True, size=(batch_size, test_size))
        boot_c_inds = np.random.choice(control_inds_array, replace=True, size=(batch_size, control_size))
        
        boot_MT = message_array[boot_t_inds]
        boot_MC = message_control[boot_c_inds]
        boot_OT = n_test[boot_t_inds]
        boot_OC = n_control[boot_c_inds]
        
        theta_asterisk = theta_func(
            boot_MT, boot_MC, boot_OT, boot_OC
        ).ravel()
        assert len(theta_asterisk) == batch_size
        theta_asterisk_array = np.concatenate([theta_asterisk_array, theta_asterisk])
    left_theta_asterisk, right_theta_asterisk = np.quantile(theta_asterisk_array, [alpha / 2, 1 - alpha / 2])

    percentile_left_bound, percentile_right_bound = left_theta_asterisk, right_theta_asterisk
    return percentile_left_bound, percentile_right_bound
    

In [52]:
bad_cnt_correct_bootstrap = 0

mc_size = 1000
for _ in range(mc_size):
    test_sample = generate_group_sample(group_size)
    control_sample = generate_group_sample(group_size)

  
    left_bound_correct, right_bound_correct = ratio_bootstrap(
        test_sample['message_array'], 
        control_sample['message_array'], 
        test_sample['O_array'], 
        control_sample['O_array']
    )
   
    bad_cnt_correct_bootstrap += ((left_bound_correct > 0) | (right_bound_correct < 0))

corr_fpr = round(bad_cnt_correct_bootstrap / mc_size, 4)
corr_left, corr_right = proportion_confint(count = bad_cnt_correct_bootstrap, 
                                           nobs = mc_size, alpha=0.05, method='wilson')
print(f"FPR, correct bootstrap: {round(corr_fpr, 4)},"\
      f" [{round(corr_left, 4)}, {round(corr_right, 4)}]")

FPR, correct bootstrap: 0.046, [0.0347, 0.0608]


Note: denominator can change in treatment group, so it's important to check that

Method 2: Linearization

Linearization is a process of transforming data in such way that:


1) adjustment is co-directional


2) works with users

from https://github.com/avito-tech/applied_statistics/blob/main/Ratio%20metrics.ipynb by using Taylor Expansion:
$$ \widehat{Z}_i = \frac{\overline{X}}{\overline{Y}}+\frac{1}{\overline{Y}}(X_i-\frac{\overline{X}}{\overline{Y}}Y_i) $$

In [53]:
def linearization(num, denom):
    x = np.mean(num)
    y = np.mean(denom)
    result = x / y + (1 / y) * (num - (x / y) * denom)
    return result

In [59]:
def generate_group_sample_equal_cr(group_size, p=0.1):
        
    O_array = stats.norm(loc=50, scale=5).rvs(size=group_size).astype(int)
    
    p_array = stats.norm(loc=p, scale=p / 10).rvs(group_size)
    p_array = np.maximum(p_array, 0)
    
    M_array = (O_array * p_array).astype(int)
    return {
        "O_array": O_array,
        "message_array": M_array
    }

In [60]:
bad_cnt_linearisation = 0
bad_cnt_bootstrap = 0
mc_size = 1000

for _ in range(mc_size):
    test_sample = generate_group_sample_equal_cr(group_size)
    control_sample = generate_group_sample_equal_cr(group_size)
    test_Z = linearization(test_sample['O_array'], test_sample['message_array'])
    control_Z = linearization(control_sample['O_array'], control_sample['message_array'])

    # Запускаю критерий и строю доверительный интервал
    left_bound, right_bound = ratio_bootstrap(
        test_sample['O_array'], 
        control_sample['O_array'], 
        test_sample['message_array'], 
        control_sample['message_array']
    )
    pvalue_linearise = stats.ttest_ind(test_Z, control_Z, equal_var=False, alternative='two-sided').pvalue
    
   
    bad_cnt_linearisation += (pvalue_linearise < 0.05)
    bad_cnt_bootstrap += ((left_bound > 0) | (right_bound < 0))


bootstrap_fpr = round(bad_cnt_bootstrap / mc_size, 4)
bootstrap_left, bootstrap_right = proportion_confint(count = bad_cnt_bootstrap, 
                                               nobs = mc_size, alpha=0.05, method='wilson')
print(f"FPR, bootstrap: {round(bootstrap_fpr, 4)},"\
      f" [{round(bootstrap_left, 4)}, {round(bootstrap_right, 4)}]")

linearise_fpr = round(bad_cnt_linearisation / mc_size, 4)
linearise_left, linearise_right = proportion_confint(count = bad_cnt_linearisation, 
                                           nobs = mc_size, alpha=0.05, method='wilson')
print(f"FPR, linearisation: {round(linearise_fpr, 4)},"\
      f" [{round(linearise_left, 4)}, {round(linearise_right, 4)}]")

FPR, bootstrap: 0.058, [0.0451, 0.0742]
FPR, linearisation: 0.052, [0.0399, 0.0676]


In [62]:
bad_cnt = 0
for i in range(n):
    test_sample = generate_group_sample_equal_cr(group_size)
    control_sample = generate_group_sample_equal_cr(group_size)
    test_Z = linearization(test_sample['O_array'], test_sample['message_array'])
    control_Z = linearization(control_sample['O_array'], control_sample['message_array'])

    pvalue_linearise = stats.ttest_ind(test_Z, control_Z, equal_var=False, alternative='two-sided').pvalue
    bad_cnt += (pvalue_linearise <= alpha)
print("FPR:", bad_cnt / n)

FPR: 0.037


Fine and fast, nothing to say :)

We also have a different way of linearization
$$ R_c = \frac{\sum_uX_c(u)}{\sum_uY_c(u)}, \:l(u)=X(u)-R_c\cdot Y(u) $$

In [64]:
def linearization(num, denom, k):
    return num - denom * k

In [ ]:
bad_cnt = 0
for i in range(n):
    test_sample = generate_group_sample_equal_cr(group_size)
    control_sample = generate_group_sample_equal_cr(group_size)

    k = np.sum(control_sample['O_array']) / np.sum(control_sample['message_array'])
    test_Z = linearization(test_sample['O_array'], test_sample['message_array'], k)
    control_Z = linearization(control_sample['O_array'], control_sample['message_array'], k)

    pvalue_linearise = stats.ttest_ind(test_Z, control_Z, equal_var=False, alternative='two-sided').pvalue
    bad_cnt += (pvalue_linearise <= alpha)
print("FPR:", bad_cnt / n)

FPR: 0.049


If data is not drawn from normal distribution we can assume that CLT holds and use Z-test.


Method 3: Delta-method


If we have large number of observations or mde is small we can use Delta-method